# Train the Q10 causal control critic

Matched control for Q50. Each action input is the **10 actions actually executed before replanning**; rewards span those same 10 environment steps and the target bootstraps at the next saved planning-boundary state. Architecture and full-run optimizer schedule match Q50 except for action length. Uncertainty is not used.

Run once with `RUN_MODE = 'smoke'`. If all ten updates and both validation passes finish, change only that line to `'full'` for the fixed 8,000-update run.

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SNAPSHOT_ID = 'PASTE_PCPCDS_SNAPSHOT_ID'  # use the same snapshot as Q50
RUN_MODE = 'smoke'  # change to 'full' only after the smoke run succeeds
MICRO_BATCH_SIZE = 16  # effective batch remains 64 via accumulation
CACHE_DOWNLOAD_WORKERS = 4  # reuses Q50 source cache in the same runtime
CACHE_ROOT = '/content/qplanning_cache'  # fast local runtime disk
OUTPUT_ROOT = '/content/drive/MyDrive/pnp_qplanning_corrector'
assert RUN_MODE in ('smoke', 'full')
print({'critic': 'Q10-causal', 'run_mode': RUN_MODE,
       'micro_batch_size': MICRO_BATCH_SIZE,
       'cache_download_workers': CACHE_DOWNLOAD_WORKERS, 'snapshot_id': SNAPSHOT_ID})

In [ ]:
from pnp.qplanning_critic import run_qplanning_training_test

report = run_qplanning_training_test(
    snapshot_id=SNAPSHOT_ID, horizon=10, run_mode=RUN_MODE,
    cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT,
    micro_batch_size=MICRO_BATCH_SIZE,
    cache_download_workers=CACHE_DOWNLOAD_WORKERS, resume=True)

In [ ]:
import pandas as pd
display(pd.DataFrame([{
    'critic': f"Q{report['horizon']}",
    'mode': report['run_mode'],
    'train_windows': report['train_windows'],
    'validation_windows': report['validation_windows'],
    **report['validation'],
}]))
print('checkpoint:', report['final_checkpoint'])